# ChuckleNet WavLM + Prosody Extraction - 555 Videos

**Runtime:** ~2-3 hours on T4 GPU
**Audio:** 621 files (~5GB) from Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
print('Drive mounted!')

In [ ]:
!apt-get install -y ffmpeg
!pip install -q transformers librosa scikit-learn

import os, json, time
import numpy as np
import torch, torch.nn as nn
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import librosa
from pathlib import Path

SR = 16000; BATCH = 16; SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Load utterances
!wget -q -O /tmp/utterances_clean.jsonl.gz https://github.com/Das-rebel/chuck-audio-notebooks/raw/main/utterances_clean.jsonl.gz
!gunzip -f /tmp/utterances_clean.jsonl.gz
utterances = [json.loads(l) for l in open('/tmp/utterances_clean.jsonl')]
print(f'Loaded {len(utterances)} utterances')
vtt_data = {}
for u in utterances:
    vid = u['video_id']
    if vid not in vtt_data: vtt_data[vid] = []
    vtt_data[vid].append(u)
print(f'Unique videos: {len(vtt_data)}')

In [ ]:
# Download audio (4.9GB, ~5 min)
!pip install -q gdown
!gdown 1jHy11LdU0bv8ra-Lj9xhpKT4yXE3L1Tp -O /tmp/vtt_audio_local.tar.gz
!tar -xzf /tmp/vtt_audio_local.tar.gz -C /tmp/
print('Audio extracted!')

In [ ]:
# Build audio file map
audio_base = Path('/tmp/vtt_audio_local')
audio_files = {f.stem: str(f) for f in audio_base.glob('*') if f.suffix in ['.m4a', '.mp3', '.wav']}
print(f'Found {len(audio_files)} audio files')
covered = sum(1 for vid in vtt_data if vid in audio_files)
print(f'Videos with audio: {covered}/{len(vtt_data)}')

In [ ]:
# Load WavLM
print('Loading WavLM...')
from transformers import WavLMModel
model = WavLMModel.from_pretrained('microsoft/wavlm-base-plus')
model = model.to(device).eval()
print('Model loaded!')

In [ ]:
def extract_prosody(audio_path, start, end, sr=SR):
    try:
        y, sr = librosa.load(audio_path, sr=sr, mono=True, offset=start, duration=end-start)
        if len(y) < sr * 0.05: return np.zeros(21, dtype=np.float32)
        feats = []
        try:
            f0, _, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
            f0v = f0[~np.isnan(f0)]
            feats.extend([np.mean(f0v), np.std(f0v), np.max(f0v), np.min(f0v), np.median(f0v)] if len(f0v)>0 else [0]*5)
        except: feats.extend([0]*5)
        rms = librosa.feature.rms(y=y)[0]
        feats.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms)])
        spec = [librosa.feature.spectral_centroid(y=y, sr=sr)[0], librosa.feature.spectral_bandwidth(y=y, sr=sr)[0], librosa.feature.spectral_flatness(y=y)[0], librosa.feature.zero_crossing_rate(y)[0]]
        feats.extend([np.mean(s[0]) for s in spec] + [np.max(spec[-1])])
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=7)
        feats.extend([np.mean(mfccs[i]) for i in range(7)])
        return np.array(feats, dtype=np.float32)
    except: return np.zeros(21, dtype=np.float32)

def extract_wavlm(audio_path, start, end):
    try:
        y, sr = librosa.load(audio_path, sr=SR, mono=True, offset=start, duration=end-start)
        if len(y) < SR * 0.05: return np.zeros(768, dtype=np.float32)
        y_t = torch.tensor(y).unsqueeze(0).to(device)
        with torch.no_grad(): emb = model(y_t).last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
        return emb
    except: return np.zeros(768, dtype=np.float32)

In [ ]:
# Extract features
print(f'Processing {len(utterances)} utterances...')
all_emb, all_pro, all_lbl, all_uid = [], [], [], []
t0 = time.time()
for i, u in enumerate(utterances):
    vid = u['video_id']
    if vid not in audio_files: continue
    s, e, lbl = u['start'], u['end'], u['label']
    all_emb.append(extract_wavlm(audio_files[vid], s, e))
    all_pro.append(extract_prosody(audio_files[vid], s, e))
    all_lbl.append(lbl); all_uid.append(f'{vid}_{s:.2f}_{e:.2f}')
    if (i+1) % 1000 == 0:
        print(f'{(i+1)/len(utterances)*100:.1f}% - {(i+1)/(time.time()-t0)*3600:.0f}/hr - ETA:{(len(utterances)-i-1)/(i+1)*(time.time()-t0)/60:.0f}min')
print(f'Done! {len(all_emb)} utterances in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Save
np.savez('/content/gdrive/MyDrive/wavlm_prosody_555.npz',
    embeddings=np.array(all_emb, dtype=np.float32),
    prosody=np.array(all_pro, dtype=np.float32),
    labels=np.array(all_lbl, dtype=np.int64),
    uids=np.array(all_uid))
print('Saved to Google Drive!')

In [ ]:
# Train sanity check
from sklearn.metrics import f1_score, precision_score, recall_score
X = np.hstack([np.array(all_emb), StandardScaler().fit_transform(np.array(all_pro))])
y = np.array(all_lbl)
Xtr, Xv, ytr, yv = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f'Train: {len(Xtr)}, Val: {len(Xv)}, Pos rate: {y.mean()*100:.1f}%')

class Clf(nn.Module):
    def __init__(self): super().__init__()
        self.fc = nn.Sequential(nn.Linear(789, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, 2))
    def forward(self, x): return self.fc(x)

m = Clf().to(device)
criterion = nn.CrossEntropyLoss()
opt = torch.optim.AdamW(m.parameters(), lr=1e-3)

Xt_t = torch.tensor(Xtr, dtype=torch.float32).to(device)
yt_t = torch.tensor(ytr, dtype=torch.long).to(device)

for ep in range(5):
    m.train()
    for i in range(0, len(Xt_t), BATCH):
        opt.zero_grad()
        loss = criterion(m(Xt_t[i:i+BATCH]), yt_t[i:i+BATCH])
        loss.backward(); opt.step()
    m.eval()
    with torch.no_grad(): vp = m(torch.tensor(Xv, dtype=torch.float32).to(device)).argmax(1).cpu().numpy()
    print(f'Epoch {ep+1}: Val F1={f1_score(yv, vp):.4f}')